# LeiGS 2026 — Proctor Prediction Challenge · v3 (PINN)
**Goal**: Predict `proctor_mdd_g_cm3` (MDD) and `proctor_owc_pct` (OWC) from soil classification features.

**v3 approach**: Physics-Informed Neural Network (PINN) that encodes geotechnical compaction physics directly into the training loss.

## Physics embedded in the model
| Constraint | Equation | Source |
|---|---|---|
| Zero-Air-Voids saturation line | ρd ≤ ρs / (1 + w·ρs/ρw) | Classical compaction theory |
| Air voids at Proctor optimum | na = 1 − (ρd/ρs)(1 + w·ρs) ∈ [2%, 20%] | Empirical compaction data |
| Hard output bounds | MDD ∈ [1.4, 2.5] g/cm³, OWC ∈ [2, 35]% | Physical plausibility |

**Metric**: NMAE = 0.5×(MAE_mdd/IQR_mdd) + 0.5×(MAE_owc/IQR_owc)

## Section 0: Setup & Imports

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # suppress OMP double-init on macOS

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
np.random.seed(42)
torch.manual_seed(42)

DEVICE = (
    torch.device('mps')  if torch.backends.mps.is_available() else
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('cpu')
)
print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}')

IQR_MDD = 0.198
IQR_OWC = 3.860
RHO_W   = 1.0
MDD_MIN, MDD_MAX = 1.40, 2.50
OWC_MIN, OWC_MAX = 2.00, 35.0

PyTorch 2.12.1  |  Device: mps


## Section 1: Data Loading

In [2]:
train_raw  = pd.read_csv('../data/train.csv')
test_raw   = pd.read_csv('../data/test.csv')
sample_sub = pd.read_csv('../sample_submission.csv')
print(f'Train: {train_raw.shape}  |  Test: {test_raw.shape}')
train_raw.head(3)

Train: (201, 25)  |  Test: (87, 23)


,id,psd_size_at_d10_mm,psd_size_at_d20_mm,psd_size_at_d30_mm,psd_size_at_d40_mm,psd_size_at_d50_mm,psd_size_at_d60_mm,psd_size_at_d70_mm,psd_size_at_d80_mm,psd_size_at_d90_mm,...,psd_passing_at_2mm_pct,proctor_mdd_g_cm3,proctor_owc_pct,proctor_diam_mm,grain_density_g_cm3,hyd_cond_kf_m_s,hyd_cond_hyd_gradient,atterberg_liquid_limit_pct,atterberg_plastic_limit_pct,loss_on_ignition_pct
0,0,0.136773,0.315214,0.547515,0.933930,1.788142,3.313938,7.214010,12.517275,20.160863,...,51.68,2.068,8.06,150,2.650,NaN,NaN,NaN,NaN,NaN
1,1,0.014579,0.062060,0.119258,0.221877,0.423550,1.022249,2.978369,6.081199,10.622063,...,65.49,2.037,10.42,100,2.690,NaN,NaN,NaN,NaN,1.3
2,2,0.003243,0.022301,0.087024,0.156595,0.217595,0.323793,0.512853,1.546115,10.921905,...,81.40,2.051,10.94,100,2.722,1.400000e-10,30.0,26.45,13.75,NaN


## Section 2: Preprocessing

In [3]:
TARGET_COLS = ['proctor_mdd_g_cm3', 'proctor_owc_pct']
PSD_D_COLS = [
    'psd_size_at_d10_mm', 'psd_size_at_d20_mm', 'psd_size_at_d30_mm',
    'psd_size_at_d40_mm', 'psd_size_at_d50_mm', 'psd_size_at_d60_mm',
    'psd_size_at_d70_mm', 'psd_size_at_d80_mm', 'psd_size_at_d90_mm',
    'psd_size_at_d95_mm', 'psd_size_at_d98_mm'
]
HIGH_MISSING = [
    'atterberg_liquid_limit_pct', 'atterberg_plastic_limit_pct',
    'hyd_cond_kf_m_s', 'hyd_cond_hyd_gradient', 'loss_on_ignition_pct'
]

def preprocess(df, medians=None, fit=False):
    df = df.copy()
    df['psd_has_sedimentation'] = df['psd_has_sedimentation'].map(
        {True: 1, False: 0, 'True': 1, 'False': 0}).astype(float)
    for col in PSD_D_COLS + HIGH_MISSING + [
        'psd_passing_at_0_002mm_pct', 'psd_passing_at_0_063mm_pct',
        'psd_passing_at_2mm_pct', 'grain_density_g_cm3', 'proctor_diam_mm'
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in HIGH_MISSING:
        df[f'{col}_missing'] = df[col].isna().astype(int)
    if fit:
        medians = {col: df[col].median() for col in HIGH_MISSING}
    for col in HIGH_MISSING:
        df[col] = df[col].fillna(medians[col])
    return df, medians

train_pp, medians = preprocess(train_raw, fit=True)
test_pp, _        = preprocess(test_raw, medians=medians, fit=False)
print('Preprocessing complete.')

Preprocessing complete.


## Section 3: Feature Engineering

In [4]:
def engineer_features(df):
    df = df.copy()
    D10 = df['psd_size_at_d10_mm'].replace(0, np.nan)
    D30 = df['psd_size_at_d30_mm']
    D60 = df['psd_size_at_d60_mm'].replace(0, np.nan)
    df['feat_cu']     = D60 / D10
    df['feat_cc']     = (D30 ** 2) / (D60 * D10)
    df['feat_log_cu'] = np.log1p(df['feat_cu'])
    df['feat_pi']     = df['atterberg_liquid_limit_pct'] - df['atterberg_plastic_limit_pct']
    df['feat_sand_pct']   = (df['psd_passing_at_2mm_pct'] - df['psd_passing_at_0_063mm_pct']).clip(lower=0)
    df['feat_gravel_pct'] = (100 - df['psd_passing_at_2mm_pct']).clip(lower=0)
    df['feat_clay_to_fines_ratio'] = (
        df['psd_passing_at_0_002mm_pct'] / df['psd_passing_at_0_063mm_pct'].replace(0, np.nan)
    ).fillna(0)
    df['feat_soil_type'] = pd.cut(
        df['psd_passing_at_0_063mm_pct'], bins=[-1, 10, 40, 101], labels=[0, 1, 2]
    ).astype(float)
    rho_s = df['grain_density_g_cm3']
    w_est = (df['psd_passing_at_0_063mm_pct'] / 10).clip(lower=3)
    df['feat_sat_mdd_proxy'] = rho_s / (1 + (w_est / 100) * rho_s / RHO_W)
    df['feat_d50'] = df['psd_size_at_d50_mm']
    for col in PSD_D_COLS:
        df[f'log_{col}'] = np.log1p(df[col])
    df['log_hyd_cond_kf_m_s'] = np.log1p(df['hyd_cond_kf_m_s'])
    return df

train_fe = engineer_features(train_pp)
test_fe  = engineer_features(test_pp)
EXCLUDE      = ['id'] + TARGET_COLS
FEATURE_COLS = [c for c in train_fe.columns if c not in EXCLUDE]

X_train_raw = train_fe[FEATURE_COLS].values.astype(np.float32)
y_mdd       = train_fe['proctor_mdd_g_cm3'].values.astype(np.float32)
y_owc       = train_fe['proctor_owc_pct'].values.astype(np.float32)
X_test_raw  = test_fe[FEATURE_COLS].values.astype(np.float32)
rho_s_train = train_fe['grain_density_g_cm3'].values.astype(np.float32)
rho_s_test  = test_fe['grain_density_g_cm3'].values.astype(np.float32)
print(f'Features: {len(FEATURE_COLS)}  |  X_train: {X_train_raw.shape}  X_test: {X_test_raw.shape}')

Features: 49  |  X_train: (201, 49)  X_test: (87, 49)


## Section 4: Metric Helpers

In [5]:
def nmae_np(y_mdd_true, y_owc_true, y_mdd_pred, y_owc_pred):
    return (0.5 * np.mean(np.abs(y_mdd_true - y_mdd_pred)) / IQR_MDD
          + 0.5 * np.mean(np.abs(y_owc_true - y_owc_pred)) / IQR_OWC)

def nmae_torch(mdd_pred, mdd_true, owc_pred, owc_true):
    return (0.5 * torch.mean(torch.abs(mdd_pred - mdd_true)) / IQR_MDD
          + 0.5 * torch.mean(torch.abs(owc_pred - owc_true)) / IQR_OWC)

print('Metric helpers defined.')

Metric helpers defined.


---
## Section 5: Physics-Informed Loss Functions

**1. Zero-Air-Voids (ZAV) saturation constraint**
$$\\rho_{d,\\text{sat}} = \\frac{\\rho_s}{1 + w \\cdot \\rho_s / \\rho_w}$$
$$\\mathcal{L}_{\\text{ZAV}} = \\mathbb{E}\\left[\\max\\left(0,\\ \\hat{\\rho}_d - 0.99\\,\\rho_{d,\\text{sat}}\\right)^2\\right]$$

**2. Air-voids fraction at Proctor optimum** (empirically 2%–20%)
$$n_a = 1 - \\frac{\\hat{\\rho}_d}{\\rho_s}\\left(1 + \\frac{\\hat{w}}{100}\\,\\rho_s\\right)$$
$$\\mathcal{L}_{\\text{AV}} = \\mathbb{E}\\left[\\max(0,\\ 0.02 - n_a)^2 + \\max(0,\\ n_a - 0.20)^2\\right]$$

In [6]:
def zav_loss(mdd_pred, owc_pred, grain_density, safety=0.99):
    w    = owc_pred / 100.0
    zav  = grain_density / (1.0 + w * grain_density / RHO_W)
    viol = torch.relu(mdd_pred - safety * zav)
    return (viol ** 2).mean()

def air_voids_loss(mdd_pred, owc_pred, grain_density, na_min=0.02, na_max=0.20):
    w    = owc_pred / 100.0
    na   = 1.0 - (mdd_pred / grain_density) * (1.0 + w * grain_density)
    low  = torch.relu(na_min - na) ** 2
    high = torch.relu(na - na_max) ** 2
    return (low + high).mean()

def physics_residuals_np(mdd_pred, owc_pred, grain_density):
    w   = owc_pred / 100.0
    zav = grain_density / (1.0 + w * grain_density / RHO_W)
    na  = 1.0 - (mdd_pred / grain_density) * (1.0 + w * grain_density)
    return {
        'ZAV violations (%)':  round(float(np.mean(mdd_pred > 0.99 * zav)) * 100, 1),
        'Air-void < 2% (%)':   round(float(np.mean(na < 0.02))  * 100, 1),
        'Air-void > 20% (%)':  round(float(np.mean(na > 0.20))  * 100, 1),
        'Mean air-void (%)':   round(float(na.mean()) * 100, 1),
        'Median air-void (%)': round(float(np.median(na)) * 100, 1),
    }

print('Physics loss functions defined.')

Physics loss functions defined.


---
## Section 6: PINN Architecture
Shared trunk MLP (SiLU activations, BatchNorm, Dropout) with two separate prediction heads.
Output activations are **sigmoid-scaled** to physical bounds — hard constraints by construction.

In [7]:
class ProctorPINN(nn.Module):
    def __init__(self, n_features, hidden=(128, 64, 32), dropout=0.35):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            in_dim = h
        self.trunk    = nn.Sequential(*layers)
        self.head_mdd = nn.Linear(in_dim, 1)
        self.head_owc = nn.Linear(in_dim, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='linear')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        z   = self.trunk(x)
        mdd = MDD_MIN + (MDD_MAX - MDD_MIN) * torch.sigmoid(self.head_mdd(z))
        owc = OWC_MIN + (OWC_MAX - OWC_MIN) * torch.sigmoid(self.head_owc(z))
        return mdd.squeeze(-1), owc.squeeze(-1)

_m = ProctorPINN(49).to(DEVICE)
_x = torch.randn(4, 49, device=DEVICE)
_mdd, _owc = _m(_x)
print(f'Forward pass OK — MDD: [{_mdd.min():.3f}, {_mdd.max():.3f}]  OWC: [{_owc.min():.3f}, {_owc.max():.3f}]')
print(f'Parameters: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}')
del _m, _x, _mdd, _owc

Forward pass OK — MDD: [1.666, 2.176]  OWC: [14.665, 25.576]
Parameters: 17,250


---
## Section 7: Training Loop

In [8]:
def train_pinn(
    X, y_mdd, y_owc, rho_s,
    lambda_zav=1.0, lambda_av=0.1,
    n_epochs=600, lr=3e-3, batch_size=32,
    weight_decay=5e-4, phys_warmup=100, verbose=False,
):
    model  = ProctorPINN(X.shape[1]).to(DEVICE)
    opt    = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched  = CosineAnnealingLR(opt, T_max=n_epochs, eta_min=lr * 0.01)
    X_t    = torch.tensor(X,     device=DEVICE)
    mdd_t  = torch.tensor(y_mdd, device=DEVICE)
    owc_t  = torch.tensor(y_owc, device=DEVICE)
    rho_t  = torch.tensor(rho_s, device=DEVICE)
    loader = DataLoader(TensorDataset(X_t, mdd_t, owc_t, rho_t),
                        batch_size=batch_size, shuffle=True, drop_last=False)
    history = {'data': [], 'zav': [], 'av': [], 'total': []}
    for epoch in range(n_epochs):
        model.train()
        ed = ez = ea = et = nb = 0
        phys_scale = min(1.0, epoch / max(phys_warmup, 1))
        for Xb, mb, ob, rb in loader:
            mp, op = model(Xb)
            Ld = nmae_torch(mp, mb, op, ob)
            Lz = zav_loss(mp, op, rb)
            La = air_voids_loss(mp, op, rb)
            Lt = Ld + phys_scale * (lambda_zav * Lz + lambda_av * La)
            opt.zero_grad(); Lt.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            ed += Ld.item(); ez += Lz.item()
            ea += La.item(); et += Lt.item(); nb += 1
        sched.step()
        history['data'].append(ed/nb); history['zav'].append(ez/nb)
        history['av'].append(ea/nb);   history['total'].append(et/nb)
        if verbose and (epoch+1) % 100 == 0:
            print(f'  Epoch {epoch+1:4d}: NMAE={history["data"][-1]:.4f}  ZAV={history["zav"][-1]:.5f}  AV={history["av"][-1]:.5f}')
    return model, history

@torch.no_grad()
def predict_pinn(model, X):
    model.eval()
    mp, op = model(torch.tensor(X, device=DEVICE))
    return mp.cpu().numpy(), op.cpu().numpy()

print('Training helpers defined.')

Training helpers defined.


---
## Section 9: XGBoost Reference Baseline

In [ ]:
xgb_params = dict(n_estimators=500, learning_rate=0.05, max_depth=4,
                  subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                  reg_lambda=1.0, random_state=42, n_jobs=-1, verbosity=0)

def cv_xgb(X, y_mdd, y_owc, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for tr, val in kf.split(X):
        mx = xgb.XGBRegressor(**xgb_params).fit(X[tr], y_mdd[tr])
        ox = xgb.XGBRegressor(**xgb_params).fit(X[tr], y_owc[tr])
        scores.append(nmae_np(y_mdd[val], y_owc[val], mx.predict(X[val]), ox.predict(X[val])))
    return np.mean(scores), np.std(scores)

print('Running XGBoost 5-fold CV...')
xgb_mean, xgb_std = cv_xgb(X_train_raw, y_mdd, y_owc)
print(f'XGBoost CV NMAE: {xgb_mean:.4f} ± {xgb_std:.4f}')

---
## Section 10: PINN Ablation — Physics Loss Weight Grid
λ=0 = pure NN (ablation control); λ>0 = physics-informed.

In [ ]:
TRAIN_KW = dict(n_epochs=600, lr=3e-3, batch_size=32, weight_decay=5e-4, phys_warmup=100)

def cv_pinn(X_raw, y_mdd, y_owc, rho_s, lambda_zav, lambda_av, n_splits=5, **kw):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for tr, val in kf.split(X_raw):
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X_raw[tr]).astype(np.float32)
        Xva = sc.transform(X_raw[val]).astype(np.float32)
        m, _ = train_pinn(Xtr, y_mdd[tr], y_owc[tr], rho_s[tr],
                          lambda_zav=lambda_zav, lambda_av=lambda_av, **kw)
        mp, op = predict_pinn(m, Xva)
        scores.append(nmae_np(y_mdd[val], y_owc[val], mp, op))
    return np.mean(scores), np.std(scores)

lambda_grid = [
    (0.0,  0.0),
    (0.5,  0.05),
    (1.0,  0.10),
    (5.0,  0.50),
    (10.0, 1.00),
]

print('Running PINN ablation (5-fold CV per λ)...')
print(f'{"Config":<25} {"CV NMAE":>10} {"Std":>8}')
print('-' * 46)
ablation_results = []
for lz, la in lambda_grid:
    label = f'λ_ZAV={lz}, λ_AV={la}'
    mean, std = cv_pinn(X_train_raw, y_mdd, y_owc, rho_s_train, lz, la, **TRAIN_KW)
    ablation_results.append({'label': label, 'lambda_zav': lz, 'lambda_av': la,
                              'cv_mean': mean, 'cv_std': std})
    tag = ' ← pure NN' if lz == 0 else ''
    print(f'{label:<25}  {mean:.4f}  ± {std:.4f}{tag}')
print('-' * 46)
print(f'XGBoost reference:         {xgb_mean:.4f}  ± {xgb_std:.4f}')
df_ablation = pd.DataFrame(ablation_results)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(df_ablation))
colors = ['#d62728' if r == 0 else '#1f77b4' for r in df_ablation['lambda_zav']]
ax.bar(x, df_ablation['cv_mean'], yerr=df_ablation['cv_std'],
       capsize=5, color=colors, edgecolor='k', linewidth=0.5)
ax.axhline(xgb_mean, color='green', linestyle='--', lw=1.5, label=f'XGBoost ({xgb_mean:.4f})')
ax.axhspan(xgb_mean - xgb_std, xgb_mean + xgb_std, alpha=0.12, color='green')
ax.set_xticks(x)
ax.set_xticklabels(df_ablation['label'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('CV NMAE (lower = better)')
ax.set_title('PINN Ablation: Physics Loss Weight vs CV NMAE')
ax.legend(); plt.tight_layout(); plt.show()
best_row = df_ablation.loc[df_ablation['cv_mean'].idxmin()]
print(f'Best PINN config: {best_row["label"]}  NMAE={best_row["cv_mean"]:.4f}')

---
## Section 10: Training Curves — Best PINN Config
Fit a fresh `StandardScaler` on all 201 training samples (no CV fold boundary),
then train the best PINN config on the full dataset. This is the only place the
global scaler lives — CV folds handle their own scaling internally.

In [ ]:
best_lz = best_row['lambda_zav']
best_la = best_row['lambda_av']

# Scaler fit on ALL training data — only for the final model, never seen by cv_pinn folds
scaler_final = StandardScaler()
X_train_sc   = scaler_final.fit_transform(X_train_raw).astype(np.float32)
X_test_sc    = scaler_final.transform(X_test_raw).astype(np.float32)

print(f'Training best PINN (λ_ZAV={best_lz}, λ_AV={best_la}) on full train...')
best_model, history = train_pinn(
    X_train_sc, y_mdd, y_owc, rho_s_train,
    lambda_zav=best_lz, lambda_av=best_la, verbose=True, **TRAIN_KW
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history['data'],  label='Data (NMAE)', lw=1.5)
axes[0].plot(history['total'], label='Total loss',  lw=1.5, ls='--')
axes[0].set_title('Training Loss Curves'); axes[0].legend()
axes[1].plot(history['zav'], label='ZAV loss',      color='coral',  lw=1.5)
axes[1].plot(history['av'],  label='Air-voids loss', color='purple', lw=1.5)
axes[1].set_title('Physics Residual Losses'); axes[1].legend()
plt.suptitle(f'Best PINN: λ_ZAV={best_lz}, λ_AV={best_la}', fontsize=12)
plt.tight_layout(); plt.show()

---
## Section 12: Physics Residual Analysis
Compare PINN vs XGBoost physics compliance on training data.

In [ ]:
mdd_pinn_tr, owc_pinn_tr = predict_pinn(best_model, X_train)
xgb_mdd_full = xgb.XGBRegressor(**xgb_params).fit(X_train_raw, y_mdd)
xgb_owc_full = xgb.XGBRegressor(**xgb_params).fit(X_train_raw, y_owc)
mdd_xgb_tr   = xgb_mdd_full.predict(X_train_raw)
owc_xgb_tr   = xgb_owc_full.predict(X_train_raw)

for label, mp, op in [('Ground truth', y_mdd, y_owc),
                       ('PINN', mdd_pinn_tr, owc_pinn_tr),
                       ('XGBoost', mdd_xgb_tr, owc_xgb_tr)]:
    print(f'\n[{label}]')
    for k, v in physics_residuals_np(mp, op, rho_s_train).items():
        print(f'  {k}: {v}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, mdd_pred, owc_pred, label, color in [
    (axes[0], y_mdd,       y_owc,       'Ground truth', 'steelblue'),
    (axes[1], mdd_pinn_tr, owc_pinn_tr, 'PINN',         'darkorange'),
    (axes[2], mdd_xgb_tr,  owc_xgb_tr,  'XGBoost',      'seagreen'),
]:
    w_p   = owc_pred / 100.0
    zav_p = rho_s_train / (1.0 + w_p * rho_s_train / RHO_W)
    ax.scatter(owc_pred, mdd_pred, c=color, alpha=0.6, s=25, edgecolors='k', lw=0.2)
    sort_idx = np.argsort(owc_pred)
    ax.plot(owc_pred[sort_idx], zav_p[sort_idx]*0.99, 'r--', lw=1.5, label='ZAV 99%')
    n_viol = int((mdd_pred > 0.99 * zav_p).sum())
    ax.set_xlabel('OWC (%)'); ax.set_ylabel('MDD (g/cm³)')
    ax.set_title(f'{label}\nViolations: {n_viol}/{len(mdd_pred)}')
    ax.legend(fontsize=8)
plt.suptitle('MDD vs ZAV Saturation Line', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
def air_voids(mdd, owc, rho_s):
    return (1.0 - (mdd / rho_s) * (1.0 + (owc/100.0) * rho_s)) * 100

na_gt   = air_voids(y_mdd,       y_owc,       rho_s_train)
na_pinn = air_voids(mdd_pinn_tr, owc_pinn_tr, rho_s_train)
na_xgb  = air_voids(mdd_xgb_tr,  owc_xgb_tr,  rho_s_train)

fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(-5, 30, 40)
ax.hist(na_gt,   bins=bins, alpha=0.5, label='Ground truth', color='steelblue', edgecolor='k', lw=0.3)
ax.hist(na_pinn, bins=bins, alpha=0.5, label='PINN',          color='darkorange', edgecolor='k', lw=0.3)
ax.hist(na_xgb,  bins=bins, alpha=0.5, label='XGBoost',       color='seagreen',  edgecolor='k', lw=0.3)
ax.axvline(2,  color='red',    ls='--', lw=1.5, label='na_min=2%')
ax.axvline(20, color='purple', ls='--', lw=1.5, label='na_max=20%')
ax.set_xlabel('Air-void fraction na (%)'); ax.set_ylabel('Count')
ax.set_title('Air-void distribution — PINN should stay within [2%, 20%]')
ax.legend(); plt.tight_layout(); plt.show()

---
## Section 13: Comparison Table

In [ ]:
comparison = pd.DataFrame(
    [{'Model': 'XGBoost', 'CV NMAE': xgb_mean, 'CV Std': xgb_std}]
    + [{'Model': r['label'], 'CV NMAE': r['cv_mean'], 'CV Std': r['cv_std']}
       for r in ablation_results]
).sort_values('CV NMAE')

print('=== Model Comparison ===')
print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['green' if 'XGB' in m else ('red' if 'λ_ZAV=0' in m else 'steelblue')
          for m in comparison['Model']]
bars = ax.barh(comparison['Model'], comparison['CV NMAE'],
               xerr=comparison['CV Std'], capsize=4,
               color=colors, edgecolor='k', lw=0.5, alpha=0.8)
ax.set_xlabel('CV NMAE (lower = better)')
ax.set_title('PINN vs XGBoost — 5-fold CV NMAE')
for bar, val in zip(bars, comparison['CV NMAE']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

---
## Section 14: PINN + XGBoost Ensemble

In [ ]:
def cv_ensemble(X_raw, y_mdd, y_owc, rho_s, lambda_zav, lambda_av,
                w_pinn=0.5, n_splits=5, **kw):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for tr, val in kf.split(X_raw):
        sc   = StandardScaler()
        Xtr  = sc.fit_transform(X_raw[tr]).astype(np.float32)
        Xva  = sc.transform(X_raw[val]).astype(np.float32)
        pm, _ = train_pinn(Xtr, y_mdd[tr], y_owc[tr], rho_s[tr],
                           lambda_zav=lambda_zav, lambda_av=lambda_av, **kw)
        mp_p, op_p = predict_pinn(pm, Xva)
        mx = xgb.XGBRegressor(**xgb_params).fit(X_raw[tr], y_mdd[tr])
        ox = xgb.XGBRegressor(**xgb_params).fit(X_raw[tr], y_owc[tr])
        mp_ens = w_pinn * mp_p + (1 - w_pinn) * mx.predict(X_raw[val])
        op_ens = w_pinn * op_p + (1 - w_pinn) * ox.predict(X_raw[val])
        scores.append(nmae_np(y_mdd[val], y_owc[val], mp_ens, op_ens))
    return np.mean(scores), np.std(scores)

print('PINN + XGBoost ensemble sweep (w_pinn)...')
ens_results = []
for wp in [0.2, 0.4, 0.5, 0.6, 0.8]:
    m, s = cv_ensemble(X_train_raw, y_mdd, y_owc, rho_s_train,
                        best_lz, best_la, w_pinn=wp, **TRAIN_KW)
    ens_results.append({'w_pinn': wp, 'cv_mean': m, 'cv_std': s})
    print(f'  w_pinn={wp:.1f}: {m:.4f} ± {s:.4f}')
df_ens = pd.DataFrame(ens_results)
best_ens = df_ens.loc[df_ens['cv_mean'].idxmin()]
print(f'\nBest ensemble w_pinn={best_ens["w_pinn"]}  NMAE={best_ens["cv_mean"]:.4f}')

---
## Section 15: Final Prediction & Submission

In [ ]:
all_nmae = {
    'XGBoost':           xgb_mean,
    'PINN':              best_row['cv_mean'],
    'PINN+XGB ensemble': best_ens['cv_mean'],
}
final_config = min(all_nmae, key=all_nmae.get)
print('CV NMAE summary:')
for k, v in sorted(all_nmae.items(), key=lambda x: x[1]):
    print(f'  {k:<28} {v:.4f}{" ← selected" if k==final_config else ""}')

if final_config == 'XGBoost':
    mx = xgb.XGBRegressor(**xgb_params).fit(X_train_raw, y_mdd)
    ox = xgb.XGBRegressor(**xgb_params).fit(X_train_raw, y_owc)
    pred_mdd = mx.predict(X_test_raw)
    pred_owc = ox.predict(X_test_raw)
elif final_config == 'PINN':
    print('Training final PINN on full train set...')
    fp, _ = train_pinn(X_train, y_mdd, y_owc, rho_s_train,
                       lambda_zav=best_lz, lambda_av=best_la, verbose=True, **TRAIN_KW)
    pred_mdd, pred_owc = predict_pinn(fp, X_test)
else:
    wp = best_ens['w_pinn']
    print(f'Training PINN+XGB ensemble (w_pinn={wp})...')
    fp, _ = train_pinn(X_train, y_mdd, y_owc, rho_s_train,
                       lambda_zav=best_lz, lambda_av=best_la, verbose=True, **TRAIN_KW)
    mp_p, op_p = predict_pinn(fp, X_test)
    mx = xgb.XGBRegressor(**xgb_params).fit(X_train_raw, y_mdd)
    ox = xgb.XGBRegressor(**xgb_params).fit(X_train_raw, y_owc)
    pred_mdd = wp * mp_p + (1-wp) * mx.predict(X_test_raw)
    pred_owc = wp * op_p + (1-wp) * ox.predict(X_test_raw)

In [ ]:
def apply_sat(mdd, owc, rho_s, safety=0.99):
    w = owc / 100.0
    return np.minimum(mdd, rho_s / (1.0 + w * rho_s / RHO_W) * safety)

pred_mdd = apply_sat(pred_mdd, pred_owc, rho_s_test)
print('Test physics residuals:')
for k, v in physics_residuals_np(pred_mdd, pred_owc, rho_s_test).items():
    print(f'  {k}: {v}')

submission = pd.DataFrame({'id': test_raw['id'],
                           'proctor_owc_pct': pred_owc,
                           'proctor_mdd_g_cm3': pred_mdd})
assert list(submission.columns) == ['id', 'proctor_owc_pct', 'proctor_mdd_g_cm3']
assert len(submission) == len(test_raw)
assert (submission['proctor_mdd_g_cm3'] > 1.4).all()
assert (submission['proctor_mdd_g_cm3'] < 2.5).all()
assert (submission['proctor_owc_pct']   > 2.0).all()
assert (submission['proctor_owc_pct']   < 35.0).all()

ts = datetime.now().strftime('%Y%m%d_%H%M')
out = f'../submissions/submission_{ts}_v3.csv'
submission.to_csv(out, index=False)
print(f'\nAll checks passed. Saved: {out}')
submission.describe().round(3)

---
## Section 16: Summary

In [ ]:
print('='*70)
print('v3 PINN Summary')
print('='*70)
print('\n[Architecture]  Shared MLP trunk (128→64→32, SiLU+BN+Dropout)')
print('               + separate MDD/OWC heads with sigmoid-scaled outputs')
print('\n[Physics loss]  L_ZAV: squared-hinge on ZAV saturation constraint')
print('               L_AV:  squared-hinge on air-void range [2%, 20%]')
print('               Curriculum warmup over first 100 epochs')
print(f'\n[Best λ]        ZAV={best_lz}, AV={best_la}')
print('\n[CV NMAE]')
for k, v in sorted(all_nmae.items(), key=lambda x: x[1]):
    print(f'  {k:<28} {v:.4f}{" ← selected" if k==final_config else ""}')
print('\n[Test physics compliance]')
for k, v in physics_residuals_np(pred_mdd, pred_owc, rho_s_test).items():
    print(f'  {k}: {v}')
print('='*70)